<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-13-industry-and-capstone/lesson-13.4-software-engineering/notebooks/exercises-13_4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 13.4: GenAI in Software Engineering

*Module 13 · Industry Applications & Capstone*

> AI doesn't replace engineers — it makes them 3-5x faster. This lesson demonstrates code generation, automated code review, test generation, documentation, and the coding agent loop: plan, code, test, fix. Plus: integrating AI into CI/CD pipelines.

`Code Generation` · `Code Review` · `Test Gen` · `Coding Agents` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-13.4.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Code Generation — From Prompt to Production Code — `01_code_generation.py`
2. Step 2 — Coding Agent — Claude Code / Aider / Cursor Patterns — `02_coding_agent.py`
3. Step 3 — CI/CD Integration — AI in Your Pipeline — `03_cicd_ai.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Code Generation — From Prompt to Production Code

Generate functions, classes, and modules from natural language specs.

**`01_code_generation.py`** · _Block 1 of 3_

CODE GENERATION — LLM as coding partner


In [ ]:
# CODE GENERATION — LLM as coding partner
import google.generativeai as genai, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-pro")

def generate_code(spec, language="python"):
    prompt = f"""Generate production-quality {language} code for:
{spec}
Requirements: type hints, docstrings, error handling, unit test.
Return ONLY code, no explanation."""
    return model.generate_content(prompt).text

def review_code(code):
    prompt = f"""Review this code for: bugs, security issues, performance,
readability, best practices. Rate each 1-5. Suggest fixes.
```
{code}
```"""
    return model.generate_content(prompt).text

def generate_tests(code):
    prompt = f"""Generate comprehensive pytest tests for this code.
Include: happy path, edge cases, error cases, boundary conditions.
```
{code}
```"""
    return model.generate_content(prompt).text

def generate_docs(code):
    prompt = f"""Generate: 1) Module docstring, 2) README section with usage examples,
3) API reference in markdown. For this code:
```
{code}
```"""
    return model.generate_content(prompt).text

print("AI-Powered Software Engineering:\n")
print("  generate_code(spec)  → production code from natural language")
print("  review_code(code)    → bugs, security, performance audit")
print("  generate_tests(code) → pytest suite with edge cases")
print("  generate_docs(code)  → docstrings + README + API reference")
print("\n  Workflow: spec → generate → review → test → document")
print("  Human: architect, review, approve. AI: draft, test, document.")


> **What just happened?** Four functions that cover the entire software lifecycle: generate from spec, review for bugs, generate tests, generate docs. The human architects and approves. The AI drafts, tests, and documents. Together: 3-5x faster than either alone.


### Step 2 · Coding Agent — Claude Code / Aider / Cursor Patterns

Autonomous coding agents that edit files, run tests, and iterate.

**`02_coding_agent.py`** · _Block 2 of 3_

CODING AGENT PATTERN — Edit, test, iterate loop


In [ ]:
# CODING AGENT PATTERN — Edit, test, iterate loop

class CodingAgent:
    \"\"\"Autonomous coding agent: plan, code, test, fix loop.\"\"\"
    def __init__(self, llm_fn, max_iterations=5):
        self.llm = llm_fn
        self.max_iter = max_iterations
        self.history = []

    def solve(self, task):
        # Step 1: Plan
        plan = self.llm(f"Create a step-by-step plan for: {task}")
        self.history.append({"step":"plan", "output":plan})

        # Step 2: Generate code
        code = self.llm(f"Implement this plan:\n{plan}\nReturn only Python code.")
        self.history.append({"step":"code", "output":code})

        # Step 3: Generate + run tests
        tests = self.llm(f"Write pytest tests for:\n{code}")

        # Step 4: Fix loop
        for i in range(self.max_iter):
            result = self._run_tests(code, tests)
            if result["passed"]:
                self.history.append({"step":"done", "iterations":i+1})
                return code
            # Fix based on errors
            code = self.llm(f"Fix this code based on test errors:\n{code}\nErrors:\n{result['errors']}")
        return code

    def _run_tests(self, code, tests):
        # In production: subprocess.run(['pytest', ...]) 
        return {"passed": True, "errors": ""}

print("Coding Agent Pattern:\n")
print("  1. PLAN:  Break task into steps")
print("  2. CODE:  Generate implementation")
print("  3. TEST:  Generate + run tests")
print("  4. FIX:   If tests fail, analyze errors, fix code, re-test")
print("  5. DONE:  All tests pass (max 5 iterations)")
print("\n  Tools: Claude Code, Aider, Cursor, Copilot Workspace")
print("  Key: human reviews EVERY merge. Agent codes, human approves.")


### Step 3 · CI/CD Integration — AI in Your Pipeline

Auto-review PRs, generate changelogs, suggest reviewers, detect issues.

**`03_cicd_ai.py`** · _Block 3 of 3_

AI IN CI/CD — Automated PR review + changelog


In [ ]:
# AI IN CI/CD — Automated PR review + changelog

class AIPRReviewer:
    \"\"\"AI-powered pull request reviewer.\"\"\"
    def __init__(self, llm_fn):
        self.llm = llm_fn

    def review_diff(self, diff):
        prompt = f"""Review this git diff. Return JSON:
{{"summary": "...", "issues": [{{"file":"...","line":N,"severity":"high|med|low","issue":"...","fix":"..."}}],
"security": [...], "tests_needed": [...], "approval": "approve|request_changes"}}
Diff:
{diff}"""
        return self.llm(prompt)

    def generate_changelog(self, commits):
        prompt = f"""Generate a changelog from these commits. Group by:
Added, Changed, Fixed, Security. Use past tense.
Commits: {commits}"""
        return self.llm(prompt)

print("AI in CI/CD Pipeline:\n")
print("  On PR opened:  AI reviews diff, flags issues, suggests fixes")
print("  On merge:      AI generates changelog entry")
print("  On deploy:     AI summarizes changes for stakeholders")
print("  Nightly:       AI scans for dependency vulnerabilities")
print("\n  GitHub Actions: add AI review step after lint + test")
print("  Rule: AI flags, human decides. Never auto-merge from AI.")


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
